# Signpost-D gated encoder–decoder: the decoder reads the case as text, the encoder adds a gated reading

Option 8 of the Signpost-D plan (see `experiments/modernbert_router_ledger.md`, "the plan of options"). It keeps the total under 0.8 B parameters:

```text
state ─► ModernBERT-base (+LoRA) ─► per-token states ─► MLP bridge ─► one soft token per state token (nothing pooled)
"<Case:> [soft tokens] \n\n Question … Options A. … B. … Answer:" ─► Qwen3-0.6B (+LoRA) ─► P(" A"), P(" B"), … ─► softmax
```

- The decoder reads one sequence: a few real tokens, the bridge's soft tokens for the case, then the question and the lettered options as text. Its causal attention lets every question token see the whole case. The answer is read from the label tokens at the last position, which is how vLLM's `/v1/systemone` reads a decoder; nothing is generated.
- **09c (this notebook, `MODE = "gated"`):** the decoder reads the case **as raw text** (as the decoder-only run 9, our best calibrated) and also gets the encoder's reading: **32 summary tokens** (the encoder states averaged over 32 consecutive stretches of the case, through the bridge), multiplied by `tanh(g)` with a learned gate `g` that **starts at 0**. At step 1 the model is exactly run 9's decoder; joint training (decoder LoRA, encoder LoRA, bridge, gate) opens the gate only as far as the encoder helps. This avoids `09b`'s collapse, where ungated soft tokens swamped the decoder.

  ```text
  "Case:
" <case as text> "

Encoder reading:
" tanh(g) x [32 summary tokens] "

" <question, options> "Answer:"
  ```
- **One stage, trained jointly (as `09b`):** no reconstruction stage. The encoder LoRA, the bridge and the decoder LoRA are trained together, from the start, on the decisions themselves: soft-label cross-entropy on the label tokens plus the expected-level term for score questions. The decision loss teaches the bridge to pass on what the decisions need. `09` (two stages: reconstruction, then decisions) ran alongside; in `09` stage 1 reached only 0.51 held-out token accuracy (0.38 before) and did not click.
- **The bridge learns fastest** (`LR_BRIDGE` 1e-3), so the soft tokens become useful before the decoder learns to answer without them; the collapse check at step 400 catches a decoder that ignores the case.
- **Control (`MODE = "decoder"`):** the same decoder and LoRA reading the case as plain text, with no encoder and no stage 1. The encoder–decoder has to beat this run to show the bridge adds anything.

Measured as in `08`: our dev cases (run 6's v1 dev cases plus invoice dev cases), grounded-question tests (items with more than 26 options are skipped, because the label letters run A–Z), the benchmark test split scored once at T = 1 and with per-type temperatures fitted on our dev set, and calibration. Nothing is uploaded. Settings are in the first code cell (`MODE = "decoder"` and `RUN_NAME = "09_decoder"` for the control).

In [ ]:
# Parameters (papermill injects values after this cell)

MODE = "gated"                        # "gated": raw text + gated encoder summary (09c); "encdec": soft tokens only; "decoder": text only
ENCODER = "answerdotai/ModernBERT-base"
DECODER = "Qwen/Qwen3-0.6B"           # ungated; a gated decoder (e.g. google/gemma-3-270m) needs HF_TOKEN in the environment
D_SET = "v1_1"
RUN_NAME = "09c_gated"
JOINT = True                          # train encoder LoRA + bridge + decoder LoRA together on the decisions (no stage 1)
STAGE1_EPOCHS = 0                     # reconstruction passes (used only if JOINT is False)
EPOCHS = 2                            # decision epochs; the best on dev KL is kept
BATCH = 8
LR_DEC, LR_ENC, LR_BRIDGE = 1e-4, 1e-4, 3e-4
N_SOFT, LR_GATE = 32, 5e-3                 # encoder summary tokens; learning rate of the gate (starts at 0)
LORA_R, LORA_ALPHA = 16, 32
SAVE_ADAPTERS = True

In [2]:
# Cell 1 — Setup: packages, fixed settings, device, progress log

!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, math, os, random, shutil, statistics, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from collections import Counter
from pathlib import Path

import pandas as pd
import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = os.environ.get("HF_TOKEN") or None     # only needed for a gated decoder; never printed or written to the notebook
if HF_TOKEN is None:
    try:                                          # Kaggle: Add-ons -> Secrets (label HF_TOKEN or hf_token, ticked for this notebook)
        from kaggle_secrets import UserSecretsClient
        for label in ("HF_TOKEN", "hf_token"):
            try:
                HF_TOKEN = UserSecretsClient().get_secret(label) or None
                break
            except Exception:
                pass
    except ImportError:
        pass
MAX_STATE_TOKENS = 1024               # encoder input (ours ~700 at most)
RECON_TOKENS = 256                    # stage 1 reconstructs the first 256 tokens of each case's text
MAX_PROMPT_TOKENS = 2048
MAX_LETTERS = 26                      # label letters A-Z
SCORE_DIST_W = 0.5
DEV_FRACTION = 0.15
SEED = 42
COLLAPSE_STEP, COLLAPSE_MARGIN = 400, 0.02
BENCH = "LocalLLaMA/typed-decisions"  # scored once at the end; never used for training or selection
LETTERS = [chr(65 + i) for i in range(MAX_LETTERS)]

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path(f"{RUN_NAME}_run.log")
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), "restart the kernel and run from the top"
api = HfApi()
REV = {ENCODER: api.model_info(ENCODER).sha, DECODER: api.model_info(DECODER, token=HF_TOKEN).sha,
       BENCH: api.dataset_info(BENCH).sha}


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"{RUN_NAME} | mode {MODE} | encoder {ENCODER} | decoder {DECODER} @ {REV[DECODER][:8]} | data {D_SET} | "
    f"torch {torch.__version__} | {DEVICE}")

[10:52:08] 09c_gated | mode gated | encoder answerdotai/ModernBERT-base | decoder Qwen/Qwen3-0.6B @ c1899de2 | data v1_1 | torch 2.11.0+cu128 | cuda:0


In [3]:
# Cell 2a — The Signpost-D cases (System One form): one item per question, with the labellers' averaged distribution as the target

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():                  # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"


def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def render(q):
    """A System One question -> ({candidate name: description}, the gold's keys in the same order). One rendering for our
    cases and the benchmark: noul as true/false with their descriptions, choice by its labels, score by its criteria
    strings, lowest first (keys "0".."n-1")."""
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        keys = ["true", "false"]
        opts = {k: cr.get(k) or "" for k in keys}
    elif q["type"] == "choice":
        keys = list(cr)
        opts = {k.replace("_", " "): v or "" for k, v in cr.items()}
    else:
        keys = [str(i) for i in range(len(cr))]
        opts = {str(c): "" for c in cr}
    assert len(opts) == len(keys), q
    return opts, keys


d_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / D_SET / "systemone.jsonl")]
def dev_split(ids):
    ids = sorted(ids)
    random.Random(SEED).shuffle(ids)
    return set(ids[: int(len(ids) * DEV_FRACTION)])


ADDED = ("I",)                            # cases added after v1 (invoices): split on their own, so v1's dev cases stay as in run 6
dev_ids = dev_split(c["id"] for c in d_cases if not c["id"].startswith(ADDED)) | \
          dev_split(c["id"] for c in d_cases if c["id"].startswith(ADDED))


def d_items(c):
    out = []
    for qid, q in c["questions"].items():
        opts, keys = render(q)
        gold = c["gold"][qid]["probabilities"]
        out.append({"task": "d_" + q["type"], "id": c["id"], "qid": qid, "domain": c["meta"]["domain"],
                    "messages": [{"role": "user", "content": state_text(c["state"])}], "current": None,
                    "question": q["instructions"], "options": opts, "levels": list(opts) if q["type"] == "score" else None,
                    "soft": {n: gold[k] for n, k in zip(opts, keys)}})
    return out


train_d = [it for c in d_cases if c["id"] not in dev_ids for it in d_items(c)]
dev_d = [it for c in d_cases if c["id"] in dev_ids for it in d_items(c)]
print(f"Signpost-D ({D_SET}): {len(d_cases)} cases | train {len(train_d)} questions | dev {len(dev_d)} questions "
      f"({len(dev_ids)} cases) | {dict(Counter(it['task'] for it in train_d))}")

Cloning into 'llms-the-hard-way'...


Signpost-D (v1_1): 1650 cases | train 6669 questions | dev 1155 questions (247 cases) | {'d_noul': 2359, 'd_choice': 2281, 'd_score': 2029}


In [4]:
# Cell 2b — Grounded-question tests (from Signpost): the general reading check

load = lambda p: json.load(open(DATA_DIR / "modernbert_router" / p))
e12 = load("choice_test/items.json")["items"]
ind_q, ind2_q = load("independent/questions.json")["items"], load("independent2/questions.json")["items"]
print(f"E12 {len(e12)} | independent questions {len(ind_q)} | independent 2 questions {len(ind2_q)} "
      f"(up to {max(len(q['options']) for q in ind2_q)} options)")

E12 170 | independent questions 300 | independent 2 questions 200 (up to 50 options)


In [5]:
# Cell 3 — Encoder, bridge, decoder; the prompt; scores over the label tokens

from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

if AMP and torch.cuda.memory_allocated(DEVICE) > 1e9:
    raise RuntimeError("the GPU already holds a model: start this notebook in a fresh kernel")
random.seed(SEED); torch.manual_seed(SEED)
dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
decoder = AutoModelForCausalLM.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN, dtype=torch.float32).to(DEVICE)
decoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
decoder.config.use_cache = False
D_DEC = decoder.config.hidden_size
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
decoder = get_peft_model(decoder, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, target_modules=PROJ))
EMBED = decoder.get_input_embeddings()
LM_HEAD = decoder.get_output_embeddings()
INNER = decoder.get_base_model().model                     # the transformer without the vocabulary head (LoRA is inside it)
LABEL_IDS = []
for L in LETTERS:
    ids = dec_tok(" " + L, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, f"label ' {L}' is not one token: {ids}"
    LABEL_IDS.append(ids[0])
LABEL_IDS = torch.tensor(LABEL_IDS, device=DEVICE)
BOS = [dec_tok.bos_token_id] if dec_tok.bos_token_id is not None else []


class Bridge(nn.Module):
    """Encoder states -> vectors in the decoder's input-embedding space: a 2-layer MLP, then RMSNorm at the embeddings' scale."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.mlp(h))


USES_ENCODER = MODE in ("encdec", "gated")
if USES_ENCODER:
    enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
    encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], dtype=torch.float32).to(DEVICE)
    encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    encoder = get_peft_model(encoder, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                                                 target_modules=["Wqkv", "Wo", "Wi"]))
    bridge = Bridge(encoder.config.hidden_size, D_DEC).to(DEVICE)
    with torch.no_grad():                                   # start the soft tokens at the scale of real token embeddings
        sample = dec_tok("The invoice total matches the order and the delivery note.", add_special_tokens=False)["input_ids"]
        bridge.norm.weight.fill_(EMBED(torch.tensor(sample, device=DEVICE)).float().pow(2).mean(-1).sqrt().mean().item())
    gate = nn.Parameter(torch.zeros(1, device=DEVICE))     # tanh(0) = 0: the encoder's reading starts switched off
else:
    encoder = bridge = gate = None
total = sum(p.numel() for p in decoder.parameters()) + (sum(p.numel() for p in encoder.parameters()) +
                                                         sum(p.numel() for p in bridge.parameters()) if encoder is not None else 0)
log(f"parameters in total {total / 1e6:.0f} M | decoder width {D_DEC} | label tokens {LABEL_IDS[:4].tolist()}...")


def question_text(item):
    """The question and the lettered options, ending at 'Answer:' (the label token comes next)."""
    names = list(item["options"])
    task = item.get("task", "d_choice")                  # grounded-test items carry no type: asked as a choice
    if task == "d_noul":
        head = f"Statement: {item['question']}\nIs the statement true or false?"
    elif task == "d_score":
        head = f"Question: {item['question']}\nLevels, lowest first:"
    else:
        head = f"Question: {item['question']}\nOptions:"
    lines = [f"{LETTERS[i]}. {n}" + (f": {item['options'][n]}" if item["options"][n] else "") for i, n in enumerate(names)]
    return head + "\n" + "\n".join(lines) + "\nAnswer:"


def tokens(text):
    return dec_tok(text, add_special_tokens=False)["input_ids"]


def soft_prefix(states):
    """Case texts -> one tensor of soft tokens per case (encoder states of its real tokens, through the bridge)."""
    t = enc_tok(states, padding=True, truncation=True, max_length=MAX_STATE_TOKENS, return_tensors="pt").to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    v = bridge(h.float())
    return [v[i, : int(t["attention_mask"][i].sum())] for i in range(len(states))]


def soft_summary(states):
    """Case texts -> per case, N_SOFT summary tokens: encoder states averaged over consecutive stretches of the case,
    through the bridge, times tanh(gate)."""
    t = enc_tok(states, padding=True, truncation=True, max_length=MAX_STATE_TOKENS, return_tensors="pt").to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    out = []
    for i in range(len(states)):
        hi = h[i, : int(t["attention_mask"][i].sum())].float()
        v = torch.stack([c.mean(0) for c in torch.tensor_split(hi, min(N_SOFT, len(hi)))])
        out.append(bridge(v) * torch.tanh(gate))
    return out


def run_decoder(pieces):
    """pieces: per example, a list of parts, each either a list of token ids or a tensor of soft tokens [n, D_DEC].
    -> the hidden state at each example's last position [B, D_DEC] (right padding; causal attention)."""
    seqs = []
    for parts in pieces:
        embs = [EMBED(torch.tensor(p, device=DEVICE)) if isinstance(p, list) else p for p in parts if len(p)]
        seqs.append(torch.cat(embs, 0))
    lens = [len(s) for s in seqs]
    assert max(lens) <= MAX_PROMPT_TOKENS, f"prompt of {max(lens)} tokens"
    x = torch.zeros(len(seqs), max(lens), D_DEC, device=DEVICE)
    mask = torch.zeros(len(seqs), max(lens), dtype=torch.long, device=DEVICE)
    for i, s in enumerate(seqs):
        x[i, : len(s)] = s
        mask[i, : len(s)] = 1
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        h = INNER(inputs_embeds=x, attention_mask=mask).last_hidden_state
    return h[torch.arange(len(seqs), device=DEVICE), torch.tensor(lens, device=DEVICE) - 1].float()


def prompt_pieces(items, prefixes=None):
    out = []
    for i, it in enumerate(items):
        text = it["messages"][0]["content"]
        if MODE == "gated":
            out.append([BOS + tokens("Case:\n" + text + "\n\nEncoder reading:\n"), prefixes[i],
                        tokens("\n\n" + question_text(it))])
        else:
            case = prefixes[i] if prefixes is not None else tokens(text)
            out.append([BOS + tokens("Case:\n"), case, tokens("\n\n" + question_text(it))])
    return out


class Model:
    """The interface of 08's Model (scores, probs, train) for the shared evaluation code."""

    def train(self, mode=True):
        decoder.train(mode)
        if encoder is not None:
            encoder.train(mode and (STAGE == 1 or JOINT)); bridge.train(mode and (STAGE == 1 or JOINT))

    def scores(self, items):
        """-> [items, most options] logits of the label tokens (missing options -1e4)."""
        assert max(len(it["options"]) for it in items) <= MAX_LETTERS
        states = [it["messages"][0]["content"] for it in items]
        prefixes = soft_prefix(states) if MODE == "encdec" else soft_summary(states) if MODE == "gated" else None
        h = run_decoder(prompt_pieces(items, prefixes))
        logits = h @ LM_HEAD.weight[LABEL_IDS].float().T                         # [B, 26]
        K = max(len(it["options"]) for it in items)
        out = torch.full((len(items), K), -1e4, device=DEVICE)
        for i, it in enumerate(items):
            out[i, : len(it["options"])] = logits[i, : len(it["options"])]
        return out

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()


STAGE = 2
model = Model()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[10:52:32] parameters in total 760 M | decoder width 1024 | label tokens [362, 425, 356, 422]...


In [6]:
# Cell 4a — The loss: soft labels (+ an expected-level term for score questions); metrics on the dev cases
# (accuracy against the label's top option, KL, Brier, ECE)

def soft_loss(scores, items):
    """Cross-entropy against the labellers' distribution (= KL + a constant). Score questions add
    SCORE_DIST_W x |expected level - target expected level| / (levels - 1), each option weighted by its level in the
    rubric (it["levels"]), not by its place after the options are shuffled."""
    K = scores.shape[1]
    target = torch.tensor([[it["soft"][n] for n in it["options"]] + [0.0] * (K - len(it["options"])) for it in items],
                          device=scores.device)
    target = target / target.sum(1, keepdim=True)
    logp = F.log_softmax(scores, dim=1)
    loss = -(target * logp).sum(1).mean()
    is_score = torch.tensor([it["task"] == "d_score" for it in items], device=scores.device)
    if is_score.any():
        lvl = torch.tensor([[it["levels"].index(n) if it["levels"] else 0 for n in it["options"]] + [0] * (K - len(it["options"]))
                            for it in items], device=scores.device, dtype=scores.dtype)
        n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
        gap = ((logp.exp() * lvl).sum(1) - (target * lvl).sum(1)).abs() / (n_levels - 1).clamp(min=1)
        loss = loss + SCORE_DIST_W * gap[is_score].mean()
    return loss


def d_predictions(model, items, batch=16):
    """-> [(item, {option: probability})] for Signpost-D items."""
    model.train(False)
    out = []
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            p = torch.softmax(model.scores(chunk), dim=1).float().cpu()
            out += [(it, {n: p[i, j].item() for j, n in enumerate(it["options"])}) for i, it in enumerate(chunk)]
    return out


def soften(p, temperature):
    """Probabilities softened as p ** (1 / T), renormalised."""
    q = [max(v, 1e-12) ** (1 / temperature) for v in p]
    z = sum(q)
    return [v / z for v in q]


def t_of(temperature, task):
    """A temperature: one number, or one per question type ({"d_noul": T, ...})."""
    return temperature.get(task, 1.0) if isinstance(temperature, dict) else temperature


def d_metrics(preds, temperature=1.0):
    """The benchmark's four measures, against our own soft labels: accuracy (top option = the label's top option), KL from the
    label, Brier, ECE (10 bins). With a temperature (one, or one per question type), probabilities are softened."""
    rows = []
    for it, p in preds:
        q = {n: max(v, 1e-12) ** (1 / t_of(temperature, it["task"])) for n, v in p.items()}
        z = sum(q.values())
        q = {n: v / z for n, v in q.items()}
        g = it["soft"]
        ok = max(q, key=q.get) == max(g, key=g.get)
        rows.append({"task": it["task"], "ok": ok, "conf": max(q.values()),
                     "kl": sum(gv * math.log(gv / max(q[n], 1e-9)) for n, gv in g.items() if gv > 0),
                     "brier": sum((q[n] - g[n]) ** 2 for n in g)})
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    return {"all": summ(rows), **{t: summ([r for r in rows if r["task"] == t]) for t in sorted({r["task"] for r in rows})}}


def fit_temperatures(preds):
    """One temperature per question type, each minimising KL on our own dev cases of that type (never the benchmark)."""
    grid = [0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5]
    out = {}
    for task in sorted({it["task"] for it, _ in preds}):
        sub = [(it, p) for it, p in preds if it["task"] == task]
        out[task] = min(grid, key=lambda t: d_metrics(sub, t)["all"]["kl"])
    return out


def calibration(pairs):
    """pairs: [(predicted probabilities, gold probabilities)], same option order. Do the predictions sit nearer 0 and 1 than
    the gold? Top-probability histograms side by side, mean normalised entropy, and the over-certain options: predicted
    under 0.01 where the gold has 0.05 or more, with the share of the total KL they cause (scorer's floor 1e-9)."""
    edges = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0001]
    hist = lambda tops: [sum(lo <= t < hi for t in tops) for lo, hi in zip(edges, edges[1:])]
    ent = lambda v: -sum(x * math.log(x) for x in v if x > 0) / math.log(len(v)) if len(v) > 1 else 0.0
    n_opt = over = 0
    kl_all = kl_over = 0.0
    for p, g in pairs:
        for pi, gi in zip(p, g):
            n_opt += 1
            term = gi * math.log(gi / max(pi, 1e-9)) if gi > 0 else 0.0
            kl_all += term
            if pi < 0.01 and gi >= 0.05:
                over += 1
                kl_over += term
    return {"top_p_bins": [f"{lo:.2f}-{min(hi, 1):.2f}" for lo, hi in zip(edges, edges[1:])],
            "pred_top_hist": hist([max(p) for p, _ in pairs]), "gold_top_hist": hist([max(g) for _, g in pairs]),
            "pred_entropy": round(statistics.mean(ent(p) for p, _ in pairs), 4),
            "gold_entropy": round(statistics.mean(ent(g) for _, g in pairs), 4),
            "over_certain_options": over, "of_options": n_opt,
            "kl_share_from_over_certain": round(kl_over / kl_all, 4) if kl_all else 0.0}

In [7]:
# Cell 4 — Stage 1 (MODE "encdec"): the decoder reconstructs each training case's text from its soft tokens

def recon_loss(states):
    prefixes = soft_prefix(states)
    pieces, targets = [], []
    for s, p in zip(states, prefixes):
        tgt = tokens(s)[:RECON_TOKENS]
        pieces.append([BOS + tokens("Case:\n"), p, tokens("\n\nThe case, word for word:\n"), tgt])
        targets.append(tgt)
    seqs = [torch.cat([EMBED(torch.tensor(q, device=DEVICE)) if isinstance(q, list) else q for q in parts if len(q)], 0)
            for parts in pieces]
    lens = [len(s) for s in seqs]
    x = torch.zeros(len(seqs), max(lens), D_DEC, device=DEVICE)
    mask = torch.zeros(len(seqs), max(lens), dtype=torch.long, device=DEVICE)
    for i, s in enumerate(seqs):
        x[i, : len(s)] = s; mask[i, : len(s)] = 1
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        h = INNER(inputs_embeds=x, attention_mask=mask).last_hidden_state
    loss, n = 0.0, 0
    for i, tgt in enumerate(targets):                      # predict each target token from the position before it
        end = lens[i]
        pos = torch.arange(end - len(tgt) - 1, end - 1, device=DEVICE)
        logits = LM_HEAD(h[i, pos].float())
        loss = loss + F.cross_entropy(logits, torch.tensor(tgt, device=DEVICE), reduction="sum")
        n += len(tgt)
    return loss / n


def recon_accuracy(states):
    """Share of target tokens the decoder predicts exactly (teacher forced): did stage 1 'click'?"""
    with torch.no_grad():
        prefixes = soft_prefix(states)
        right = total = 0
        for s, p in zip(states, prefixes):
            tgt = tokens(s)[:RECON_TOKENS]
            parts = [BOS + tokens("Case:\n"), p, tokens("\n\nThe case, word for word:\n"), tgt]
            x = torch.cat([EMBED(torch.tensor(q, device=DEVICE)) if isinstance(q, list) else q for q in parts if len(q)], 0)[None]
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                h = INNER(inputs_embeds=x).last_hidden_state[0]
            end = x.shape[1]
            pred = LM_HEAD(h[end - len(tgt) - 1: end - 1].float()).argmax(-1)
            right += int((pred == torch.tensor(tgt, device=DEVICE)).sum()); total += len(tgt)
    return right / max(total, 1)


STAGE1_LOG = []
if USES_ENCODER and not JOINT:
    STAGE = 1
    train_states = sorted({state_text(c["state"]) for c in d_cases if c["id"] not in dev_ids})
    held_states = sorted({state_text(c["state"]) for c in d_cases if c["id"] in dev_ids})[:40]
    params = [p for p in encoder.parameters() if p.requires_grad] + list(bridge.parameters()) + \
             [p for p in decoder.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": [p for p in encoder.parameters() if p.requires_grad], "lr": LR_ENC},
                             {"params": list(bridge.parameters()), "lr": LR_BRIDGE},
                             {"params": [p for p in decoder.parameters() if p.requires_grad], "lr": LR_DEC}], weight_decay=0.0)
    steps = STAGE1_EPOCHS * -(-len(train_states) // BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / 50, max(0.0, (steps - s) / max(1, steps - 50))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log(f"stage 1: {len(train_states)} case texts, {steps} steps | reconstruction accuracy before: {recon_accuracy(held_states):.3f}")
    step, t0, run = 0, time.perf_counter(), []
    for epoch in range(1, STAGE1_EPOCHS + 1):
        model.train(True)
        order = random.Random(SEED + epoch).sample(train_states, len(train_states))
        for i in range(0, len(order), BATCH):
            loss = recon_loss(order[i:i + BATCH])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run.append(loss.item()); step += 1
            if step % 50 == 0 or step == steps:
                log(f"  stage 1 step {step}/{steps} | loss {statistics.mean(run[-50:]):.4f} | "
                    f"{(time.perf_counter() - t0) / 60:.1f} min | {gpu()}")
        acc = recon_accuracy(held_states)
        STAGE1_LOG.append({"epoch": epoch, "loss": round(statistics.mean(run[-50:]), 4), "held_out_token_accuracy": round(acc, 4)})
        log(f"stage 1 epoch {epoch} | held-out reconstruction token accuracy {acc:.3f}")
    for n, p in decoder.named_parameters():               # reset the decoder LoRA (Thanos run 16: reset beat carry)
        if "lora_B" in n:
            p.data.zero_()
    for p in list(encoder.parameters()) + list(bridge.parameters()):
        p.requires_grad_(False)                           # stage 2 trains the decoder LoRA only (as the control does)
    del opt, sched, scaler
    gc.collect(); torch.cuda.empty_cache()
    STAGE = 2
    log("stage 1 done: decoder LoRA reset; encoder and bridge frozen for stage 2")
else:
    log("no stage 1: " + ("JOINT, all parts trained together on the decisions" if USES_ENCODER else "MODE decoder"))

[10:52:33] no stage 1: JOINT, all parts trained together on the decisions


In [8]:
# Cell 5 — Stage 2: decisions on our cases (the decoder LoRA), best epoch by dev KL; collapse check as in 08

class Collapsed(Exception):
    """The predictions stayed at a uniform guess."""


def uniform_kl(items):
    return statistics.mean(sum(g * math.log(g * len(it["soft"])) for g in it["soft"].values() if g > 0) for it in items)


def lora_state():
    st = {"dec." + n: p.detach().clone() for n, p in decoder.named_parameters() if p.requires_grad}
    if USES_ENCODER and JOINT:
        st.update({"enc." + n: p.detach().clone() for n, p in encoder.named_parameters() if p.requires_grad})
        st.update({"bridge." + n: p.detach().clone() for n, p in bridge.named_parameters()})
    if MODE == "gated":
        st["gate"] = gate.detach().clone()
    return st


def train_decisions():
    per_epoch = -(-len(train_d) // BATCH)
    groups = [{"params": [p for p in decoder.parameters() if p.requires_grad], "lr": LR_DEC}]
    if USES_ENCODER and JOINT:
        groups += [{"params": [p for p in encoder.parameters() if p.requires_grad], "lr": LR_ENC},
                   {"params": list(bridge.parameters()), "lr": LR_BRIDGE}]
    if MODE == "gated":
        groups.append({"params": [gate], "lr": LR_GATE, "weight_decay": 0.0})
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    total = EPOCHS * per_epoch
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    check = dev_d[:300]
    best, step, t0, history, run = (float("inf"), None, 0), 0, time.perf_counter(), [], []
    log(f"stage 2: {per_epoch} batches per epoch, {total} steps | trainable {sum(p.numel() for p in params) / 1e6:.1f} M")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        d = train_d[:]
        random.Random(SEED + epoch).shuffle(d)
        for i in range(0, len(d), BATCH):
            items = [dict(it) for it in d[i:i + BATCH]]
            for it in items:                              # choice and noul options shuffled; score levels keep their order
                if it["task"] != "d_score":
                    names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            loss = soft_loss(model.scores(items), items)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run.append(loss.item()); step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  epoch {epoch} step {step}/{total} | loss {statistics.mean(run[-100:]):.4f} | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}" +
                    (f" | gate tanh(g) {torch.tanh(gate).item():+.4f}" if MODE == "gated" else ""))
            if step == COLLAPSE_STEP:
                kl, base = d_metrics(d_predictions(model, check))["all"]["kl"], uniform_kl(check)
                log(f"  collapse check at step {step}: dev KL {kl:.4f} vs uniform {base:.4f}")
                if kl > base - COLLAPSE_MARGIN:
                    raise Collapsed(f"dev KL {kl:.4f} at step {step}, uniform guess {base:.4f}")
                model.train(True)
        dev = d_metrics(d_predictions(model, dev_d))["all"]
        history.append({"epoch": epoch, **dev})
        log(f"epoch {epoch} done | dev {dev}")
        if dev["kl"] < best[0]:
            best = (dev["kl"], lora_state(), epoch)
    by_name = {"dec." + n: p for n, p in decoder.named_parameters()}
    if USES_ENCODER:
        by_name.update({"enc." + n: p for n, p in encoder.named_parameters()})
        by_name.update({"bridge." + n: p for n, p in bridge.named_parameters()})
    if MODE == "gated":
        by_name["gate"] = gate
    for n, v in best[1].items():
        by_name[n].data.copy_(v)
    minutes = (time.perf_counter() - t0) / 60
    log(f"stage 2 done in {minutes:.1f} min | best epoch {best[2]} (dev KL {best[0]:.4f})")
    return {"train_minutes": minutes, "best_epoch": best[2], "history": history}


trained = train_decisions()

[10:52:33] stage 2: 834 batches per epoch, 1668 steps | trainable 15.3 M


/tmp/ipykernel_116/2084944244.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


[10:57:03]   epoch 1 step 100/1668 | loss 1.1755 | ~70.6 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0272
[11:01:39]   epoch 1 step 200/1668 | loss 1.0493 | ~66.8 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0481
[11:06:19]   epoch 1 step 300/1668 | loss 0.9770 | ~62.8 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0388
[11:10:54]   epoch 1 step 400/1668 | loss 0.9413 | ~58.2 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0393
[11:11:19]   collapse check at step 400: dev KL 0.4136 vs uniform 0.6641
[11:15:53]   epoch 1 step 500/1668 | loss 0.9490 | ~54.5 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0410
[11:20:33]   epoch 1 step 600/1668 | loss 0.9123 | ~49.8 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0435
[11:25:05]   epoch 1 step 700/1668 | loss 0.8880 | ~45.0 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0424
[11:29:42]   epoch 1 step 800/1668 | loss 0.9024 | ~40.3 min left | GPU 3.3 GB (peak 5.0) | gate tanh(g) +0.0417
[11:32:59] epoch 1 done

In [9]:
# Cell 5a — Timing and the question test (from Signpost's 03)

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": [{"role": "user", "content": it["state"]}], "current": None,
                                    "question": it["question"], "options": {o: "" for o in it["options"]}})
        top = int(p.argmax())
        rows.append({"type": it["type"], "ok": it["options"][top] == it["answer"],
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [10]:
# Cell 5b — The grounded-question tests on a model: R = {test: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}


def evaluate(model):
    model.train(False)
    R, LAT = {}, {}
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        rows, LAT[name] = questions(model, items)
        R[f"{name} questions"] = (sum(r["ok"] for r in rows), len(rows))
        for t in TYPE:
            rs = [r for r in rows if r["type"] == t]
            if rs:
                R[f"{name} {TYPE[t]}"] = (sum(r["ok"] for r in rs), len(rs))
        log(f"{name} questions {R[f'{name} questions'][0]}/{len(rows)} | " +
            " · ".join(f"{TYPE[t]} {R[f'{name} {TYPE[t]}'][0]}/{R[f'{name} {TYPE[t]}'][1]}" for t in TYPE if f"{name} {TYPE[t]}" in R))
    return R, LAT

In [11]:
# Cell 5c — The benchmark, scored once at the end (general mode: never used for training or for any choice above).
# Scoring copied from the benchmark's bench/score_with_jobs.py (score_one, 10-bin ECE).

def bench_split(split):
    f = hf_hub_download(BENCH, f"all/{split}-00000-of-00001.parquet", repo_type="dataset", revision=REV[BENCH])
    rows = pd.read_parquet(f).to_dict("records")
    for r in rows:
        r["state"], r["questions"], r["gold"] = (json.loads(r[k]) for k in ("state", "questions", "gold"))
    return rows


def score_one(pred, gold):
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":
        pred = dict(pred)
        pred["probabilities"] = {{"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()}
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p, am_g = p.index(max(p)), g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def bench_item(case, q):
    """A benchmark question as a Signpost item, through the same render() as our own cases (state as its JSON text, as
    /v1/systemone sends it)."""
    opts, keys = render(q)
    return {"messages": [{"role": "user", "content": state_text(case["state"])}], "current": None,
            "question": q.get("instructions") or "", "options": opts}, keys


def bench_score(model, cases, temperature=1.0, pairs=None):
    """The four measures on the benchmark. temperature: one number, or one per question type. pairs (a list), if given,
    collects (prediction, gold) in gold order for calibration()."""
    model.train(False)
    rows = []
    for case in cases:
        for qid, q in case["questions"].items():
            item, keys = bench_item(case, q)
            p = soften(model.probs(item).tolist(), t_of(temperature, "d_" + q["type"]))
            pred = {"probabilities": dict(zip(keys, p))}
            ok, kl, brier, conf = score_one(pred, case["gold"][qid])
            rows.append({"type": q["type"], "workflow": case["workflow"], "ok": ok, "kl": kl, "brier": brier, "conf": conf})
            if pairs is not None:
                g = case["gold"][qid]["probabilities"]
                pairs.append(([pred["probabilities"].get(k, 0.0) for k in g], [float(v) for v in g.values()]))
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl_from_gold": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    out = {"all": summ(rows)}
    for f in ("type", "workflow"):
        out.update({f"{f}={v}": summ([r for r in rows if r[f] == v]) for v in sorted({r[f] for r in rows})})
    return out

In [12]:
# Cell 5d — Benchmark items carry their type, so the prompt asks a statement, a choice or a scale as in training

_bench_item = bench_item


def bench_item(case, q):
    item, keys = _bench_item(case, q)
    item["task"] = "d_" + q["type"]
    return item, keys

In [13]:
# Cell 6 — Every measure (as 08), plus per-question probabilities saved for model averaging

dev_preds = d_predictions(model, dev_d)
T = fit_temperatures(dev_preds)
AFTER = {"T=1": d_metrics(dev_preds), "T per type": d_metrics(dev_preds, T)}
for k, v in AFTER.items():
    log(f"after training, dev cases, {k}: " + " | ".join(f"{t} {m}" for t, m in v.items()))
DEV_PARTS = {}
for name, part in (("v1 cases (as run 6)", [(it, p) for it, p in dev_preds if not it["id"].startswith(ADDED)]),
                   ("added cases (invoices)", [(it, p) for it, p in dev_preds if it["id"].startswith(ADDED)])):
    if part:
        DEV_PARTS[name] = d_metrics(part, T)["all"]
        log(f"after training, dev {name}, T per type: {DEV_PARTS[name]}")

e12[:] = [it for it in e12 if len(it["options"]) <= MAX_LETTERS]
ind_q[:] = [it for it in ind_q if len(it["options"]) <= MAX_LETTERS]
ind2_q[:] = [it for it in ind2_q if len(it["options"]) <= MAX_LETTERS]
R, LAT = evaluate(model)
log(f"temperatures (dev): {T}")
TEST = bench_split("test")
pairs_t1, pairs_t = [], []
BENCH_RES = {"T=1": bench_score(model, TEST, 1.0, pairs_t1), "T per type": bench_score(model, TEST, T, pairs_t)}
CALIB = {"dev, T=1": calibration([([p[n] for n in it["options"]], [it["soft"][n] for n in it["options"]]) for it, p in dev_preds]),
         "benchmark, T=1": calibration(pairs_t1), "benchmark, T per type": calibration(pairs_t)}
for k, v in CALIB.items():
    log(f"calibration, {k}: {v}")
for k, v in BENCH_RES.items():
    log(f"benchmark test, {k}: {v['all']}")
    for g, m in v.items():
        if g != "all":
            print(f"  {g:38s} acc {m['accuracy']:.3f}  KL {m['kl_from_gold']:.3f}  Brier {m['brier']:.3f}  ECE {m['ece']:.3f}  (n {m['n']})")
print("benchmark reference points (accuracy): uniform 0.308 · prior 0.470 · Signpost 0.4175 · run 6 (08) 0.4365")
PROBS = {"dev": [{"id": it["id"], "qid": it["qid"], "p": [p[n] for n in it["options"]]} for it, p in dev_preds],
         "bench_t1": [p for p, _ in pairs_t1]}
json.dump({"run": RUN_NAME, "mode": MODE, "encoder": ENCODER, "decoder": DECODER, "d_set": D_SET, "revisions": REV,
           "parameters_m": round(total / 1e6), "joint": JOINT, "gate": torch.tanh(gate).item() if gate is not None else None, "stage1": STAGE1_LOG, **trained, "dev_after": AFTER, "dev_parts": DEV_PARTS, "temperature": T,
           "calibration": CALIB, "question_tests": {k: list(v) for k, v in R.items()}, "benchmark": BENCH_RES, "probs": PROBS},
          open(f"{RUN_NAME}_results.json", "w"), indent=1)
if SAVE_ADAPTERS:
    OUT = Path(f"{RUN_NAME}_adapters")
    shutil.rmtree(OUT, ignore_errors=True)
    decoder.save_pretrained(OUT / "decoder_lora")
    if encoder is not None:
        encoder.save_pretrained(OUT / "encoder_lora")
        torch.save({"bridge": bridge.state_dict(), "gate": gate.detach().cpu() if gate is not None else None,
                    "n_soft": N_SOFT, "mode": MODE}, OUT / "bridge.pt")
log(f"saved {RUN_NAME}_results.json" + (f" and {RUN_NAME}_adapters/" if SAVE_ADAPTERS else ""))

[12:14:47] after training, dev cases, T=1: all {'n': 1155, 'accuracy': 0.6892, 'kl': 0.457, 'brier': 0.297, 'ece': 0.0449} | d_choice {'n': 419, 'accuracy': 0.6826, 'kl': 0.4892, 'brier': 0.2928, 'ece': 0.0631} | d_noul {'n': 406, 'accuracy': 0.7512, 'kl': 0.3549, 'brier': 0.2729, 'ece': 0.058} | d_score {'n': 330, 'accuracy': 0.6212, 'kl': 0.5417, 'brier': 0.3319, 'ece': 0.0468}
[12:14:47] after training, dev cases, T per type: all {'n': 1155, 'accuracy': 0.6892, 'kl': 0.457, 'brier': 0.297, 'ece': 0.0449} | d_choice {'n': 419, 'accuracy': 0.6826, 'kl': 0.4892, 'brier': 0.2928, 'ece': 0.0631} | d_noul {'n': 406, 'accuracy': 0.7512, 'kl': 0.3549, 'brier': 0.2729, 'ece': 0.058} | d_score {'n': 330, 'accuracy': 0.6212, 'kl': 0.5417, 'brier': 0.3319, 'ece': 0.0468}
[12:14:47] after training, dev v1 cases (as run 6), T per type: {'n': 1056, 'accuracy': 0.6866, 'kl': 0.452, 'brier': 0.2933, 'ece': 0.0465}
[12:14:47] after training, dev added cases (invoices), T per type: {'n': 99, 'accuracy

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

[12:26:13] calibration, dev, T=1: {'top_p_bins': ['0.00-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-0.95', '0.95-1.00'], 'pred_top_hist': [220, 240, 187, 169, 189, 81, 69], 'gold_top_hist': [17, 32, 36, 51, 149, 296, 574], 'pred_entropy': 0.7036, 'gold_entropy': 0.2925, 'over_certain_options': 0, 'of_options': 3601, 'kl_share_from_over_certain': 0.0}
[12:26:13] calibration, benchmark, T=1: {'top_p_bins': ['0.00-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-0.95', '0.95-1.00'], 'pred_top_hist': [551, 441, 291, 257, 285, 166, 9], 'gold_top_hist': [366, 497, 398, 232, 203, 104, 200], 'pred_entropy': 0.7192, 'gold_entropy': 0.6391, 'over_certain_options': 4, 'of_options': 7100, 'kl_share_from_over_certain': 0.0012}
[12:26:13] calibration, benchmark, T per type: {'top_p_bins': ['0.00-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-0.95', '0.95-1.00'], 'pred_top_hist': [551, 441, 291, 257, 285, 166, 9], 'gold_top_hist': [366, 497, 398,

In [14]:
# Cell 7 — Field-flip test (benchmark train split, reading only): does the answer move when only the deciding fields change?
# Run 7 (08): agent outcome 0.758 / 60 of 60, agent action 0.163 / 0, invoice 0.005 / 0, security 0.022 / 1.

N_FLIP = 60


def flip_dist(case, qid):
    item, keys = bench_item(case, case["questions"][qid])
    return dict(zip(keys, soften(model.probs(item).tolist(), t_of(T, "d_" + case["questions"][qid]["type"]))))


def agent_variants(s):
    a, b = copy.deepcopy(s), copy.deepcopy(s)
    a["trace_summary"].update(constraint_violations=0, irreversible_actions=0, tool_errors=0)
    b["trace_summary"].update(constraint_violations=2, irreversible_actions=2, tool_errors=3)
    return a, b


def invoice_variants(s):
    a, b = copy.deepcopy(s), copy.deepcopy(s)
    for v, factor in ((a, 1.0), (b, 1.15)):
        for il, pl in zip(v["invoice"]["lines"], v["purchase_order"]["lines"]):
            il["unit_usd"] = round(pl["unit_usd"] * factor, 2)
            il["total_usd"] = round(il["unit_usd"] * il["qty"], 2)
        v["invoice"]["total_usd"] = round(sum(l["total_usd"] for l in v["invoice"]["lines"]), 2)
    return a, b


def security_variants(s):
    a, b = copy.deepcopy(s), copy.deepcopy(s)
    a["principal"]["mfa_enrolled"], b["principal"]["mfa_enrolled"] = True, False
    a["context"].update(source_on_allowlist=True, change_window_active=True, asset_criticality="low")
    b["context"].update(source_on_allowlist=False, change_window_active=False, asset_criticality="crown_jewel")
    return a, b


FLIP_RES = {}
train_cases = bench_split("train")
model.train(False)
for wf, make, qids in [("agent_trace_observability", agent_variants, ["action", "outcome"]),
                       ("invoice_processing", invoice_variants, ["disposition"]),
                       ("security_incidents", security_variants, ["disposition"])]:
    pool = [c for c in train_cases if c["workflow"] == wf][:N_FLIP]
    for qid in qids:
        tvs, flips = [], 0
        for c in pool:
            try:
                sa, sb = make(c["state"])
            except (KeyError, TypeError):
                continue
            da, db = flip_dist({**c, "state": sa}, qid), flip_dist({**c, "state": sb}, qid)
            tvs.append(0.5 * sum(abs(da[k] - db[k]) for k in da))
            flips += max(da, key=da.get) != max(db, key=db.get)
        FLIP_RES[f"{wf}/{qid}"] = {"cases": len(tvs), "mean_tv": round(statistics.mean(tvs), 3), "top_changed": flips}
        log(f"field-flip {wf}/{qid}: {FLIP_RES[f'{wf}/{qid}']}")
res = json.load(open(f"{RUN_NAME}_results.json"))
res["field_flip"] = FLIP_RES
json.dump(res, open(f"{RUN_NAME}_results.json", "w"), indent=1)

all/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  599kB            

all/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

[12:26:31] field-flip agent_trace_observability/action: {'cases': 60, 'mean_tv': 0.225, 'top_changed': 34}
[12:26:49] field-flip agent_trace_observability/outcome: {'cases': 60, 'mean_tv': 0.652, 'top_changed': 60}
[12:27:07] field-flip invoice_processing/disposition: {'cases': 60, 'mean_tv': 0.011, 'top_changed': 0}
[12:27:25] field-flip security_incidents/disposition: {'cases': 60, 'mean_tv': 0.098, 'top_changed': 5}
